In [ ]:
"""Day 26 / Session 1 - Lab Assignment 1: Extractive Summarization by Word-Frequency Scoring + ROUGE-1."""
import re
from collections import Counter


def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def split_sentences(text):
    parts = re.split(r"[.!?]+", str(text))
    return [p.strip() for p in parts if p.strip()]


def word_frequencies(text):
    counts = Counter(tokenize(text))
    if not counts:
        return {}
    top = max(counts.values())
    return {word: count / top for word, count in counts.items()}     # the most frequent word maps to 1.0


def score_sentences(sentences, freqs):
    return [float(sum(freqs.get(token, 0.0) for token in tokenize(s))) for s in sentences]


def extractive_summary(text, k):
    sentences = split_sentences(text)
    if k <= 0 or not sentences:
        return []
    scores = score_sentences(sentences, word_frequencies(text))
    ranked = sorted(range(len(sentences)), key=lambda i: (-scores[i], i))     # ties -> the earlier sentence
    chosen = sorted(ranked[:k])                                               # back to the original order
    return [sentences[i] for i in chosen]


def rouge1(candidate, reference):
    cand = Counter(tokenize(candidate))
    ref = Counter(tokenize(reference))
    n_cand, n_ref = sum(cand.values()), sum(ref.values())
    if n_cand == 0 or n_ref == 0:
        return 0.0
    overlap = sum(min(count, ref[token]) for token, count in cand.items())
    precision, recall = overlap / n_cand, overlap / n_ref
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)
